![logo](./img/TheBridge_RL.png)

# Taxi Autónomo (Smartcab)

Implementación actualizada a Gymnasium. Se respetan `terminated` y `truncated`, se fijan semillas y se cierran los entornos. Taxi utiliza la versión disponible de Gymnasium (v4 en el entorno validado).

## Contenidos

* [Objetivo](#Objetivo)  
* [Recompensas (Rewards)](#Recompensas-(Rewards))  
* [Espacio de Estados (State Space)](#Espacio-de-Estados-(State-Space))  
* [Action space](#Action-space)  
* [Implementacion](#Implementacion)  


### Objetivo
  
[al indice](#Contenidos)  

El trabajo del Smartcab es recoger al pasajero en un lugar y dejarlo en otro. Algunos detalles que nos encantaría que nuestro Smartcab tenga en cuenta serían:

* Dejar al pasajero en la ubicación correcta.  
* Ahorrar tiempo al pasajero dedicando el mínimo tiempo posible para dejarlo.  
* Cuidar la seguridad del pasajero y respetar las normas de tráfico.


### Recompensas (Rewards)  
[al indice](#Contenidos)  


* El agente debería recibir una alta recompensa positiva por una entrega del cliente exitosa porque este comportamiento es de los más importantes que queremos que aprenda.
* El agente debería ser penalizado si intenta dejar a un pasajero en destinos incorrectos.
* El agente debería recibir una ligera recompensa negativa por no llegar a destino después de cada intervalo de tiempo. "Ligera" negativa porque preferiríamos que nuestro agente llegue tarde en lugar de hacer movimientos erróneos tratando de llegar al destino lo más rápido posible."

Para nuestro pequeño ejercicio vamos a establecer las siguientes "recompensas":
* Recibimos +20 puntos por un traslado exitoso.  
* Perdemos 1 punto por cada intervalo de tiempo que tarda.  
* También hay una penalización de 10 puntos por acciones de recogida y dejada ilegales. 



### Espacio de Estados (State Space)  
[al indice](#Contenidos)  


El __Espacio de Estados__ es el conjunto de todas las posibles situaciones en las que nuestro taxi podría estar. El estado además debe contener información útil que el agente necesite para tomar la acción correcta.

<img src="./img/Reinforcement_Learning_Taxi_Env.png" alt="drawing" width="600"/>

Supongamos que Smartcab es el único vehículo en este circuito de aprendizaje. Nuestro circuito está dividido en __una cuadrícula de 5x5__, lo que nos da __25 posibles ubicaciones__ para el taxi (posiciones (0,0) a (4,4)). Estas 25 ubicaciones __son una parte de nuestro espacio de estados__. Observa que la __ubicación actual__ de nuestro taxi es la __coordenada (3, 1)__.

También puedes ver que hay __cuatro (4) ubicaciones__ en las que podemos __recoger y dejar__ a un pasajero: R, G, Y, B o [(0,0), (0,4), (4,0), (4,3)] en coordenadas (fila, columna).  

También debemos tener en cuenta un (1) estado adicional del pasajero de estar dentro del taxi.



¿cómo codificaríamos el estado anterior de la forma más compacta:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from IPython.display import display
from rl_support import animate,episode,evaluate,train_taxi,load_or_train,cache_path,TAXI_ID
SEED=42
env=gym.make(TAXI_ID,render_mode='ansi');env.reset(seed=19)
state=env.unwrapped.encode(3,1,2,0)
assert state==328
print(state,list(env.unwrapped.decode(state)))

328 [3, 1, 2, 0]


Teniendo en cuenta lo anterior, ¿cuántos estados posibles hay en nuestro State Space?

In [2]:
print('Estados:',5*5*5*4,'Generalización:',25*26*25)
print('Hay 300 estados iniciales válidos: 25*4*3. Con pasajero a bordo se añaden 100 estados no terminales y existen 4 terminales de entrega. El destino queda fijo en cada episodio.')

Estados: 500 Generalización: 16250
Hay 300 estados iniciales válidos: 25*4*3. Con pasajero a bordo se añaden 100 estados no terminales y existen 4 terminales de entrega. El destino queda fijo en cada episodio.


El entorno nos devolverá un índice entre 0 y 499 para representar el estado al que se llega después de ejecutar una acción (invocando el método step())

Extra: 
* Si quisieramos generalizar y tener como destino y origen cualquiera de las cuadrículas de nuestro circuito, ¿cuántos estados tendría nuestro espacio de estados?
* Considerando los 500 estados que tenemos, ¿cuántos realmente se pueden visitar por partida? (Pista: Si al inicializar el entorno el pasajero está ya en su destino, ¿qué ocurre?)

### Action space  
[al indice](#Contenidos)  


El agente se encuentra con uno de los 500 estados y toma una acción. La acción en nuestro caso puede ser moverse en una dirección o decidir recoger/dejar a un pasajero.

En otras palabras, tenemos seis posibles acciones:
1. sur
2. norte
3. este
4. oeste
5. recoger
6. dejar


Según la ilustración, el taxi no puede realizar ciertas acciones en ciertos estados debido a las paredes (por ejemplo mover de la posición (3,1) a la (3,0)).  



<img src = "./img/Reinforcement_Learning_Taxi_Env.png" />

El entorno proporciona una penalización de -1 por cada movimiento no permitido y el taxi no se moverá a ningún lado. Como siguiente estado devuelve el mismo de partida.

### Implementacion  
[al indice](#Contenidos)  


In [3]:
state,info=env.reset(seed=19)
print(env.render());print('Estado',state,'Acciones',env.action_space,'Observaciones',env.observation_space)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+


Estado 208 Acciones Discrete(6) Observaciones Discrete(500)


* El cuadrado relleno representa el taxi, que es de color amarillo sin un pasajero y verde con un pasajero. En este caso, comenzaríamos con nuestro taxi situado en (2,0).
* La barra ("|") representa una pared que el taxi no puede cruzar.
* R, G, Y, B son las posibles ubicaciones de recogida y destino. La letra azul representa la ubicación actual de recogida del pasajero, y la letra morada es el destino actual. Es decir hay que recogerlo de Y y entregarlo en R, para este ejemplo generado al resetear el entorno con la semilla ajustada al valor 19.

In [4]:
print(list(env.unwrapped.decode(state)))

[2, 0, 2, 0]


In [5]:
print('Máscara de acciones:',info['action_mask'])

Máscara de acciones: [1 1 1 0 0 0]


Si quisieramos camibar la posición de partida de nuestro taxi y llevarlo a la de la figura, podemos moverlo con las acciones e ignorar las recompensas. Bastaría con moverlo al este y luego al sur.

In [6]:
for action in [2,0]:
 state,reward,terminated,truncated,info=env.step(action)
 print(env.render());print(state,reward)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (East)

228 -1
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (South)

328 -1


In [7]:
env.unwrapped.s=env.unwrapped.encode(3,1,2,0)
print(env.render())
env.close()

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (South)

